# Data Cleaning — TheLook Customer Analysis

**Scope:** Clean the 4 tables used for Customer Analysis.

**Input:** `../data/raw/` → **Output:** `../data/processed/`

| Table | Key cleaning steps |
|---|---|
| `order_items` | Parse datetimes, drop `inventory_item_id` |
| `orders` | Parse datetimes |
| `users` | Parse datetimes, drop PII columns, fix country names |
| `products` | Drop `name`, `sku`, `distribution_center_id`; fill brand nulls |

## 0. Imports & Load Data

In [1]:
import os
import pandas as pd
import numpy as np

RAW       = os.path.join("..", "data", "raw")
PROCESSED = os.path.join("..", "data", "processed")

order_items = pd.read_csv(os.path.join(RAW, "order_items.csv"))
orders      = pd.read_csv(os.path.join(RAW, "orders.csv"))
products    = pd.read_csv(os.path.join(RAW, "products.csv"))
users       = pd.read_csv(os.path.join(RAW, "users.csv"))

In [2]:
# Quick overview
for name, df in [("order_items", order_items), ("orders", orders),
                 ("products", products), ("users", users)]:
    print(f"{name:<15} {df.shape[0]:>9,} rows  x  {df.shape[1]} cols")

order_items       180,862 rows  x  11 cols
orders            124,814 rows  x  9 cols
products           29,120 rows  x  9 cols
users             100,000 rows  x  15 cols


## 1. Table: order_items

In [3]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 180862 entries, 0 to 180861
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 180862 non-null  int64  
 1   order_id           180862 non-null  int64  
 2   user_id            180862 non-null  int64  
 3   product_id         180862 non-null  int64  
 4   inventory_item_id  180862 non-null  int64  
 5   status             180862 non-null  str    
 6   created_at         180862 non-null  str    
 7   shipped_at         117919 non-null  str    
 8   delivered_at       63463 non-null   str    
 9   returned_at        18041 non-null   str    
 10  sale_price         180862 non-null  float64
dtypes: float64(1), int64(5), str(5)
memory usage: 25.1 MB


In [4]:
# Parse datetimes
for col in ['created_at', 'shipped_at', 'delivered_at', 'returned_at']:
    order_items[col] = pd.to_datetime(order_items[col], format='mixed', utc=True)

# Drop column not needed for analysis
order_items = order_items.drop(columns=['inventory_item_id'])

In [5]:
print("Missing values:")
print(order_items.isna().sum()[order_items.isna().sum() > 0].to_string() or "  none")
print(f"Duplicates: {order_items.duplicated().sum()}")
print("\nStatus values:", order_items['status'].unique())
print("\nSale price:")
print(order_items['sale_price'].describe().round(2))

Missing values:
shipped_at       62943
delivered_at    117399
returned_at     162821
Duplicates: 0

Status values: <ArrowStringArray>
['Cancelled', 'Complete', 'Processing', 'Returned', 'Shipped']
Length: 5, dtype: str

Sale price:
count    180862.00
mean         59.69
std          66.88
min           0.02
25%          24.95
50%          39.99
75%          69.95
max         999.00
Name: sale_price, dtype: float64


In [6]:
order_items.to_csv(os.path.join(PROCESSED, "order_items_clean.csv"), index=False)
print("✅ order_items_clean.csv saved")

✅ order_items_clean.csv saved


## 2. Table: orders

In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 124814 entries, 0 to 124813
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   order_id      124814 non-null  int64
 1   user_id       124814 non-null  int64
 2   status        124814 non-null  str  
 3   gender        124814 non-null  str  
 4   created_at    124814 non-null  str  
 5   returned_at   12439 non-null   str  
 6   shipped_at    81454 non-null   str  
 7   delivered_at  43867 non-null   str  
 8   num_of_item   124814 non-null  int64
dtypes: int64(3), str(6)
memory usage: 15.5 MB


In [8]:
# Parse datetimes
for col in ['created_at', 'returned_at', 'shipped_at', 'delivered_at']:
    orders[col] = pd.to_datetime(orders[col], format='mixed', utc=True)

In [9]:
print("Missing values (%)")
print(orders.isna().mean().mul(100).round(2).to_string())
print(f"\nDuplicates: {orders.duplicated().sum()}")
print("\nStatus values:", orders['status'].unique())
print("num_of_item:", orders['num_of_item'].describe().round(2).to_dict())

Missing values (%)
order_id         0.00
user_id          0.00
status           0.00
gender           0.00
created_at       0.00
returned_at     90.03
shipped_at      34.74
delivered_at    64.85
num_of_item      0.00

Duplicates: 0

Status values: <ArrowStringArray>
['Cancelled', 'Complete', 'Processing', 'Returned', 'Shipped']
Length: 5, dtype: str
num_of_item: {'count': 124814.0, 'mean': 1.45, 'std': 0.8, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 4.0}


In [10]:
orders.to_csv(os.path.join(PROCESSED, "orders_clean.csv"), index=False)
print("✅ orders_clean.csv saved")

✅ orders_clean.csv saved


## 3. Table: users

In [11]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              100000 non-null  int64  
 1   first_name      100000 non-null  str    
 2   last_name       100000 non-null  str    
 3   email           100000 non-null  str    
 4   age             100000 non-null  int64  
 5   gender          100000 non-null  str    
 6   state           100000 non-null  str    
 7   street_address  100000 non-null  str    
 8   postal_code     100000 non-null  str    
 9   city            99045 non-null   str    
 10  country         100000 non-null  str    
 11  latitude        100000 non-null  float64
 12  longitude       100000 non-null  float64
 13  traffic_source  100000 non-null  str    
 14  created_at      100000 non-null  str    
dtypes: float64(2), int64(2), str(11)
memory usage: 23.0 MB


In [12]:
# Parse datetime
users['created_at'] = pd.to_datetime(users['created_at'], format='mixed', utc=True)

# Drop PII and columns not needed for customer analysis
users = users.drop(columns=['first_name', 'last_name', 'email',
                             'street_address', 'postal_code',
                             'latitude', 'longitude'])

# Fix country name inconsistencies
users['country'] = users['country'].replace({
    'España': 'Spain',
    'Deutschland': 'Germany'
})

In [13]:
print("Missing values:")
print(users.isna().sum()[users.isna().sum() > 0].to_string() or "  none")
# Note: city nulls are small — analysis uses country/state level, so acceptable
print(f"\nDuplicates: {users.duplicated().sum()}")
print("\nAge:", users['age'].describe().round(2).to_dict())
print("Gender:", users['gender'].value_counts().to_dict())
print("Traffic source:", users['traffic_source'].unique())

Missing values:
city    955

Duplicates: 0

Age: {'count': 100000.0, 'mean': 41.06, 'std': 17.07, 'min': 12.0, '25%': 26.0, '50%': 41.0, '75%': 56.0, 'max': 70.0}
Gender: {'M': 50148, 'F': 49852}
Traffic source: <ArrowStringArray>
['Search', 'Display', 'Organic', 'Email', 'Facebook']
Length: 5, dtype: str


In [14]:
users.to_csv(os.path.join(PROCESSED, "users_clean.csv"), index=False)
print("✅ users_clean.csv saved")

✅ users_clean.csv saved


## 4. Table: products

In [15]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 29120 entries, 0 to 29119
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      29120 non-null  int64  
 1   cost                    29120 non-null  float64
 2   category                29120 non-null  str    
 3   name                    29118 non-null  str    
 4   brand                   29096 non-null  str    
 5   retail_price            29120 non-null  float64
 6   department              29120 non-null  str    
 7   sku                     29120 non-null  str    
 8   distribution_center_id  29120 non-null  int64  
dtypes: float64(2), int64(2), str(5)
memory usage: 4.9 MB


In [16]:
# Drop columns not needed for analysis
products = products.drop(columns=['name', 'sku', 'distribution_center_id'])

# Fill brand nulls
products['brand'] = products['brand'].fillna('Unknown')

In [17]:
print("Missing values:")
print(products.isna().sum()[products.isna().sum() > 0].to_string() or "  none")
print(f"\nDuplicates: {products.duplicated().sum()}")
print("\nCategories:", products['category'].nunique(), "unique")
print("Departments:", products['department'].unique())
print("\nPrice stats:")
print(products[['cost', 'retail_price']].describe().round(2))

Missing values:
Series([], )

Duplicates: 0

Categories: 26 unique
Departments: <ArrowStringArray>
['Women', 'Men']
Length: 2, dtype: str

Price stats:
           cost  retail_price
count  29120.00      29120.00
mean      28.48         59.22
std       30.62         65.89
min        0.01          0.02
25%       11.28         24.00
50%       19.68         39.99
75%       34.44         69.95
max      557.15        999.00


In [18]:
products.to_csv(os.path.join(PROCESSED, "products_clean.csv"), index=False)
print("✅ products_clean.csv saved")

✅ products_clean.csv saved


---

## Summary

All 4 tables cleaned and saved to `../data/processed/`:

| File | Key changes |
|---|---|
| `order_items_clean.csv` | Datetimes parsed, `inventory_item_id` dropped |
| `orders_clean.csv` | Datetimes parsed |
| `users_clean.csv` | PII dropped, country names standardised |
| `products_clean.csv` | Unused columns dropped, brand nulls filled |